# V-RAG Finetuning - LLaVA-S to LLaVA-Vrag

Finetunes LLaVA-S on the mixed V-RAG dataset so that it can read several images at once and use
retrieved reference cases when answering. The result is the model referred to as LLaVA-Vrag.

## Inputs
- The LLaVA-S LoRA adapter from the previous notebook
- `train_vrag_dataset.zip` (33,000 mixed samples) from the dataset builder
- `chexpert_cache/` for the images referenced by the dataset

## Outputs
- `llava_vrag_checkpoints.zip` containing the trained adapter: the two most recent checkpoints
  and the checkpoint with the lowest training loss

## Design notes
Training runs as a launched script rather than inline cells because it is distributed across two
GPUs, and a separate process cannot see notebook variables. The run is resumable and stops
before the platform's session limit so that a long job can continue across sessions.

## Environment and GPU check

Confirms both GPUs are visible before launching. A single visible GPU means the runtime was
configured incorrectly and the distributed launch would fail.

In [1]:
import subprocess, sys, torch
if "unsloth" not in subprocess.run([sys.executable,"-m","pip","list"],capture_output=True,text=True).stdout:
    subprocess.run("pip install -q unsloth", shell=True)
subprocess.run("pip install -q --no-deps trl peft accelerate bitsandbytes", shell=True)
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(),
      "->", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3

## Training script

Writes the complete training pipeline to a file. It contains dataset loading, the multi-image
prompt construction, model loading from either the LLaVA-S adapter or a previous checkpoint,
the training loop, and the callbacks that keep the lowest-loss checkpoint, log progress, and
stop cleanly before the session limit. Everything lives in one script because the distributed
launcher runs it as a separate process.

In [ ]:
%%writefile /kaggle/working/train.py
import os, io, re, json, csv, zipfile, random, shutil, time, math, inspect, dataclasses
from pathlib import Path
from collections import Counter
import numpy as np
from PIL import Image
import torch
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoTokenizer, TrainerCallback
from trl import SFTTrainer, SFTConfig

T_START = time.time()                                # for the wall-clock timer
RANK       = int(os.environ.get("RANK", 0))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", 1))
IS_MAIN    = RANK == 0
def log(*a):
    if IS_MAIN: print(*a, flush=True)

# ========================== CONFIG ==========================
SANITY_MODE      = False                # False = the real 11h relay run
TIME_LIMIT_HOURS = 11.0                 # stop, save, and exit before Kaggle's 12h kill
DATASET   = "/train_vrag_dataset/train_vrag_dataset.jsonl"
LLAVA_S   = "//llava-s-lora-adapter/pytorch/default/1/llava_s_adapter"
CACHE_HINT= "//cached-data-chexpert/chexpert_cache"
TOKENIZER_REF = "unsloth/llava-1.5-7b-hf-bnb-4bit"
IMG_TOKENS, MAX_SEQ_LEN = 576, 4096
LR, EPOCHS, WARMUP = 5e-5, 1, 0.03
BATCH, GRAD_ACCUM  = 2, 4
SEED = 42
WORK, OUT_DIR, BEST_DIR = "/kaggle/working", "/kaggle/working/llava_vrag", "/kaggle/working/llava_vrag_best"
ZIP_PATH   = f"{WORK}/llava_vrag_checkpoints.zip"
SAVE_STEPS, SAVE_LIMIT, LOG_STEPS = 200, 2, 10
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if SANITY_MODE:
    SUBSET, MAX_STEPS, SAVE_STEPS, LOG_STEPS, TIME_LIMIT_HOURS = 300, 30, 10, 5, 11.0
else:
    SUBSET, MAX_STEPS = None, -1        # -1 -> full epoch (~2,060 steps at eff batch 16)
log(f"[rank{RANK}] world={WORLD_SIZE} | eff batch = {BATCH}x{GRAD_ACCUM}x{WORLD_SIZE} = {BATCH*GRAD_ACCUM*WORLD_SIZE} "
    f"| time limit {TIME_LIMIT_HOURS}h | {'SANITY' if SANITY_MODE else 'REAL'}")

# ========== RESUME DETECTION (find an uploaded checkpoint) ==========
def _extract_zips_if_needed():
    """If the resume checkpoint was uploaded as a zip, unpack it (rank-0 only; fs-synced)."""
    zips = [z for z in Path("/kaggle/input").rglob("*.zip") if "checkpoint" in z.name.lower() or "vrag" in z.name.lower()]
    if not zips: return
    dst = Path(f"{WORK}/_resume")
    if LOCAL_RANK == 0:
        dst.mkdir(parents=True, exist_ok=True)
        for z in zips:
            try:
                with zipfile.ZipFile(z) as zf: zf.extractall(dst)
                log(f"[rank{RANK}] extracted resume zip: {z.name}")
            except Exception as e:
                log(f"[rank{RANK}] zip extract failed ({z.name}): {e}")
        (dst/".extracted").write_text("done")
    else:
        while not (dst/".extracted").exists(): time.sleep(2)

def find_resume():
    """Highest checkpoint-N dir under the inputs (or an extracted zip). Identified by name
       'checkpoint-N' + adapter weights -> never matches the clean LLaVA_S adapter."""
    _extract_zips_if_needed()
    roots = [Path("/kaggle/input"), Path(f"{WORK}/_resume")]
    cks = []
    for root in roots:
        if not root.exists(): continue
        for ts in root.rglob("trainer_state.json"):
            d = ts.parent
            m = re.search(r"checkpoint-(\d+)$", d.name)
            if m and (d/"adapter_model.safetensors").exists():
                cks.append((int(m.group(1)), str(d)))
    if not cks: return None, 0
    step, path = max(cks)
    return path, step

RESUME, RESUME_STEP = find_resume()
if RESUME: log(f"[rank{RANK}] RESUME from step {RESUME_STEP}: {RESUME}")
else:      log(f"[rank{RANK}] no checkpoint found — starting fresh from LLaVA_S")

# ========== cache index (fast bounded locate, manifest-driven) ==========
from tqdm.auto import tqdm
_hits = []
for _d in ("*", "*/*", "*/*/*", "*/*/*/*"):
    _hits += list(Path("/kaggle/input").glob(f"{_d}/manifest.csv"))
CACHE = next((p.parent for p in _hits if list(p.parent.glob("images_part*"))), None)
if CACHE is None:
    for _d in ("*", "*/*", "*/*/*", "*/*/*/*"):
        for _p in Path("/kaggle/input").glob(f"{_d}/chexpert_plus_full.parquet"):
            if list(_p.parent.glob("images_part*")): CACHE = _p.parent
        if CACHE: break
assert CACHE is not None, "cache dir not found under /kaggle/input"
log(f"[rank{RANK}] cache dir: {CACHE}")
parts = sorted(CACHE.glob("images_part*")); extracted = parts[0].is_dir()
man = CACHE / "manifest.csv"; key2src = {}
if man.exists():
    for row in tqdm(csv.reader(open(man)), desc=f"[rank{RANK}] cache index", disable=not IS_MAIN):
        if not row or row[0] in ("key","png_key"): continue
        k, sh = row[0], row[1]
        key2src[k] = ("file", str(CACHE/sh.replace(".zip","")/k)) if extracted else ("zip", str(CACHE/sh))
else:
    for p in parts:
        if zipfile.is_zipfile(p):
            with zipfile.ZipFile(p) as zf:
                for n in zf.namelist():
                    if n.endswith(".png"): key2src[n] = ("zip", str(p))
log(f"[rank{RANK}] cache index: {len(key2src):,} images")

# ========== load jsonl + token budget + per-session reshuffle ==========
rows = [json.loads(l) for l in open(DATASET) if l.strip()]
_tok = AutoTokenizer.from_pretrained(TOKENIZER_REF)
def seq_len(r):
    t = r["prompt"].replace("<image>","")
    return r["n_img"]*IMG_TOKENS + len(_tok(t, add_special_tokens=False).input_ids) + \
           len(_tok(str(r["target"]), add_special_tokens=False).input_ids)
rows = [r for r in tqdm(rows, desc=f"[rank{RANK}] tokenise", disable=not IS_MAIN) if seq_len(r) <= MAX_SEQ_LEN]
# reshuffle by (seed + resume step) so each session sees a different order (we use ignore_data_skip)
random.Random(SEED + RESUME_STEP).shuffle(rows)
if SUBSET: rows = rows[:SUBSET]
log(f"[rank{RANK}] samples: {len(rows):,} | tasks {dict(Counter(r['task'] for r in rows))}")
_miss = {k for r in rows for k in r["images"]} - set(key2src)
assert not _miss, f"missing images: {list(_miss)[:3]}"

# ========== dataset (multi-image; split prompt on <image>) ==========
class VRAGDataset(TorchDataset):
    def __init__(self, rows): self.rows=rows; self._h={}; self._pid=None
    def _zf(self, p):
        if self._pid != os.getpid(): self._h, self._pid = {}, os.getpid()
        if p not in self._h: self._h[p] = zipfile.ZipFile(p)
        return self._h[p]
    def _img(self, k):
        kind, loc = key2src[k]
        im = Image.open(loc) if kind=="file" else Image.open(io.BytesIO(self._zf(loc).read(k)))
        return im.convert("RGB")
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]; imgs=[self._img(k) for k in r["images"]]
        pr = r["prompt"].split("<image>"); assert len(pr)==len(imgs)+1
        content=[]
        for j,part in enumerate(pr):
            if part.strip(): content.append({"type":"text","text":part})
            if j<len(imgs): content.append({"type":"image","image":imgs[j]})
        return {"messages":[{"role":"user","content":content},
                            {"role":"assistant","content":[{"type":"text","text":str(r["target"])}]}]}
train_dataset = VRAGDataset(rows)

# ========== model: load from RESUME if present, else LLaVA_S ==========
MODEL_SRC = RESUME if RESUME else LLAVA_S
log(f"[rank{RANK}] loading model from {MODEL_SRC} (silent ~1-2 min) ...")
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_SRC, load_in_4bit=True, use_gradient_checkpointing="unsloth")
FastVisionModel.for_training(model)
tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"[rank{RANK}] trainable params: {tp:,}"); assert tp > 0, "adapter not trainable"

# collator + max-length lever (else image tokens get truncated)
try: model.max_seq_length = MAX_SEQ_LEN
except Exception: pass
for _t in (tokenizer, getattr(tokenizer,"tokenizer",None)):
    if _t is not None:
        try: _t.model_max_length = MAX_SEQ_LEN
        except Exception: pass
_sig = inspect.signature(UnslothVisionDataCollator.__init__).parameters
_kw = {"max_seq_length":MAX_SEQ_LEN} if "max_seq_length" in _sig else \
      ({"max_length":MAX_SEQ_LEN} if "max_length" in _sig else {})
collator = UnslothVisionDataCollator(model, tokenizer, **_kw)

# ========== callbacks: best-ckpt (rank0) · 11h timer · log ==========
class KeepBestCheckpoint(TrainerCallback):
    def __init__(self, d): self.d, self.best, self.latest = d, float("inf"), None
    def on_log(self, a, s, c, logs=None, **k):
        if logs and "loss" in logs: self.latest = logs["loss"]
    def on_save(self, a, s, c, **k):
        if not s.is_world_process_zero: return
        if self.latest is None or self.latest >= self.best: return
        src = os.path.join(a.output_dir, f"checkpoint-{s.global_step}")
        if os.path.isdir(src):
            self.best = self.latest
            if os.path.exists(self.d): shutil.rmtree(self.d)
            shutil.copytree(src, self.d)
            json.dump({"step":s.global_step,"loss":self.best}, open(os.path.join(self.d,"best_info.json"),"w"))
            print(f"   best loss {self.best:.4f} @ step {s.global_step}", flush=True)

class TimeLimit(TrainerCallback):
    def __init__(self, hours): self.deadline = T_START + hours*3600; self.hit = False
    def on_step_end(self, a, s, c, **k):
        if not self.hit and time.time() > self.deadline:
            self.hit = True
            c.should_save = True; c.should_training_stop = True   # save at stop point, then exit
            print(f"\n[rank{RANK}] {TIME_LIMIT_HOURS}h reached at step {s.global_step} — saving & stopping.", flush=True)
        return c

class LogProgress(TrainerCallback):
    """Background-friendly: prints a compact line each logging step (no tqdm spam in the log)."""
    def on_log(self, a, s, c, logs=None, **k):
        if not IS_MAIN or not logs or "loss" not in logs: return
        el = time.time() - T_START; done, total = s.global_step, s.max_steps or 0
        eta = el/max(done,1)*(total-done) if total else 0
        left = self.__class__.deadline - time.time() if hasattr(self.__class__,"deadline") else 0
        print(f"[{time.strftime('%H:%M:%S')}] step {done}/{total} | loss {logs['loss']:.4f} | lr {logs.get('learning_rate',0):.2e} "
              f"| {el/3600:.2f}h elapsed | ~{eta/3600:.1f}h left in run | {left/3600:.2f}h to 11h-stop", flush=True)
LogProgress.deadline = T_START + TIME_LIMIT_HOURS*3600

_bf16 = torch.cuda.is_bf16_supported()
_cfg = dict(output_dir=OUT_DIR, per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=WARMUP,
            num_train_epochs=EPOCHS, max_steps=MAX_STEPS, seed=SEED,
            fp16=not _bf16, bf16=_bf16, optim="adamw_8bit",
            logging_steps=LOG_STEPS, save_steps=SAVE_STEPS, save_total_limit=SAVE_LIMIT,
            report_to="none", remove_unused_columns=False, disable_tqdm=True,
            dataloader_num_workers=2, dataset_kwargs={"skip_prepare_dataset":True},
            ignore_data_skip=True,                 # don't replay seen batches on resume (we reshuffle instead)
            ddp_find_unused_parameters=(WORLD_SIZE > 1))
_fields = {f.name for f in dataclasses.fields(SFTConfig)}
if   "max_seq_length" in _fields: _cfg["max_seq_length"] = MAX_SEQ_LEN
elif "max_length"     in _fields: _cfg["max_length"]     = MAX_SEQ_LEN
args = SFTConfig(**{k:v for k,v in _cfg.items() if k in _fields})

_tkw = dict(model=model, data_collator=collator, train_dataset=train_dataset, args=args,
            callbacks=[KeepBestCheckpoint(BEST_DIR), TimeLimit(TIME_LIMIT_HOURS), LogProgress()])
if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters: _tkw["processing_class"]=tokenizer
else: _tkw["tokenizer"]=tokenizer
trainer = SFTTrainer(**_tkw)

# ========== train (resume restores optimizer + step + LR schedule) ==========
log(f"[rank{RANK}] {'resuming @ step '+str(RESUME_STEP) if RESUME else 'fresh start'} | training up to the "
    f"{TIME_LIMIT_HOURS}h limit ...")
res = trainer.train(resume_from_checkpoint=RESUME)

# ========== save + zip (rank 0) ==========
if IS_MAIN:
    model.save_pretrained(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
    srcs = [Path(OUT_DIR)] + ([Path(BEST_DIR)] if Path(BEST_DIR).exists() else [])
    files = [f for s in srcs for f in s.rglob("*") if f.is_file()]
    with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:
        for f in files: z.write(f, f.relative_to(WORK))
    done = res.global_step >= (trainer.state.max_steps or 0)
    print("\n" + "="*64, flush=True)
    if done:
        print(f"TRAINING COMPLETE at step {res.global_step}. Final model in {OUT_DIR}.", flush=True)
    else:
        print(f"STOPPED at step {res.global_step}/{trainer.state.max_steps} "
              f"(11h limit). NOT finished — resume next session.", flush=True)
    print(f"{ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.0f} MB) — 2 latest + 1 best.", flush=True)
    print("Upload this zip as a Kaggle Model, add it to the notebook inputs, and rerun to continue.", flush=True)
    print("="*64, flush=True)


Writing /kaggle/working/train.py


## Launch training

Starts the script across both GPUs. Each process trains on a distinct slice of the data and only
the adapter gradients are synchronised between them, which is what makes multi-GPU training
inexpensive for a LoRA run.

In [3]:
USE_DDP = False    # False = single GPU (recommended for the relay). True = 2x T4 DDP via torchrun.

import subprocess
cmd = ("torchrun --nproc_per_node=2 train.py" if USE_DDP else "python train.py")
print(f"launching: {cmd}\n" + "="*60, flush=True)
subprocess.run(f"cd /kaggle/working && {cmd}", shell=True, check=False)

launching: python train.py


Traceback (most recent call last):
  File "/kaggle/working/train.py", line 7, in <module>
    from unsloth import FastVisionModel
  File "/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py", line 1432, in <module>
    from ._gpu_init import *
  File "/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py", line 141, in <module>
    import unsloth_zoo
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/__init__.py", line 333, in <module>
    from .device_type import (
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/device_type.py", line 250, in <module>
    DEVICE_TYPE : str = get_device_type()
                        ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/device_type.py", line 235, in get_device_type
    raise NotImplementedError("Unsloth cannot find any torch accelerator? You need a GPU.")
NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.


CompletedProcess(args='cd /kaggle/working && python train.py', returncode=1)

## Inspect the output

Reports the checkpoints produced and the final training state, confirming whether the run
completed or stopped at the time limit and needs to be resumed.

In [ ]:
import os, json
from pathlib import Path
OUT, BEST, ZP = "../working/llava_vrag", "../working/llava_vrag_best", "../working/llava_vrag_checkpoints.zip"
print("latest checkpoints:", [p.name for p in sorted(Path(OUT).glob("checkpoint-*"))] if Path(OUT).exists() else "none")
if Path(BEST).exists():
    bi = json.load(open(Path(BEST)/"best_info.json")); print(f"best: loss {bi['loss']:.4f} @ step {bi['step']}")
print(f"zip : {ZP} ({os.path.getsize(ZP)/1e6:.0f} MB)" if os.path.exists(ZP) else "zip: not found — did Cell 3 finish?")
print("\nCreate/Update a Kaggle Model from this zip, add it to the inputs, and rerun to continue.")

latest checkpoints: none
zip: not found — did Cell 3 finish?

Create/Update a Kaggle Model from this zip, add it to the inputs, and rerun to continue.


## Result of this stage

The checkpoint archive contains the LLaVA-Vrag adapter. Before evaluation it is converted into a
single adapter directory containing the adapter weights and tokenizer files.